# Model a push-pull traveling-wave MZM

Build a Mach–Zehnder modulator, add differential/common-mode electrodes, then simulate a **56 Gbaud optical link** with input and output grating couplers. Start with binary modulation, then compare PAM4 eyes with and without transmitter timing jitter.

Each PN section has junction capacitance, quadratic phase versus voltage, constant optical loss and native propagation delay. Parameters are illustrative. Use the Circulax checkout containing `DifferentialAdapter` and `mode_transmission_line`.

All component definitions, wiring and waveform utilities are included below. For a compact two-port alternative, see [the reduced RLGC line example](../electrical/rlgc_transmission_line.ipynb).

In [ ]:
import copy
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from circulax import compile_circuit
from circulax.components import DifferentialAdapter, mode_transmission_line
from circulax.components.base_component import component
from circulax.components.electronic import (
    WAVE_DC,
    WAVE_PWL,
    WAVE_SIN,
    Capacitor,
    Resistor,
    WaveformVoltageSource,
)
from circulax.components.photonic import DirectionalCoupler, Grating, OpticalSource

jax.config.update("jax_enable_x64", True)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True})
c0 = 299_792_458.0
length_m, n_sections = 2e-3, 6
dz = length_m / n_sections
n_opt, wavelength_um, optical_loss_db_cm = 4.05, 1.55, 1.0
rj_product, cj_per_m = 4.7e-3, 320e-12  # ohm m, F/m
v_bias, phase_a1, phase_a2 = -1.0, 1.2, 0.15  # V, rad/(V mm), rad/(V² mm)
source_r = load_r = 30.0  # per conductor, ohms
laser_power, grating_loss_db = 1e-3, 3.0  # W, dB per grating
quadrature_phase = -np.pi / 2
print(f"{n_sections} sections: {dz * 1e6:.1f} um, Rj={rj_product / dz:.2f} ohm, Cj={cj_per_m * dz * 1e15:.1f} fF")

## 1. Build the optical MZM

Two 50/50 couplers split and recombine the light. Terminate their unused ports with matched 1 Ω optical-envelope loads. A separate phase shifter sets the bias:

```text
                  ┌──── arm A ───────┐
input ─ splitter                     combiner ─ output
                  └──── arm B ─ bias ┘
```

Start with constant-phase arms. This wiring gives $P_\mathrm{out}/P_\mathrm{in}=[1-\cos(\theta)]/2$. Sweep the bias to verify that $\theta=-\pi/2$ sets quadrature: half transmission with maximum modulation slope.

In [ ]:
@component(ports=("p1", "p2"), states=("a1", "a2"), holomorphic=True)
def PhaseBias(signals, phase=0.0):  # noqa: N802
    """Matched, reciprocal phase-only optical element; field impedance is 1 ohm."""
    transmission = jnp.exp(1j * phase)
    b1, b2 = transmission * signals.a2, transmission * signals.a1
    return {
        "p1": signals.a1 - b1,
        "p2": signals.a2 - b2,
        "a1": signals.p1 - signals.a1 - b1,
        "a2": signals.p2 - signals.a2 - b2,
    }, {}


def optical_netlist(arm_a, arm_b):
    return {
        "instances": {
            "GND": {"component": "ground"},
            "SPLIT": {"component": "coupler", "settings": {"coupling": 0.5}},
            "COMBINE": {"component": "coupler", "settings": {"coupling": 0.5}},
            "BIAS": {"component": "bias", "settings": {"phase": quadrature_phase}},
            "DUMP_IN": {"component": "R", "settings": {"R": 1.0}},
            "DUMP_OUT": {"component": "R", "settings": {"R": 1.0}},
        },
        "connections": {
            "SPLIT,p2": "DUMP_IN,p1",
            "DUMP_IN,p2": "GND,p1",
            "DUMP_OUT,p2": "GND,p1",
            "SPLIT,p3": arm_a[0],
            "SPLIT,p4": arm_b[0],
            arm_a[1]: "COMBINE,p1",
            arm_b[1]: "BIAS,p1",
            "BIAS,p2": "COMBINE,p2",
            "COMBINE,p4": "DUMP_OUT,p1",
        },
        "ports": {"optical_in": "SPLIT,p1", "optical_out": "COMBINE,p3"},
    }

In [ ]:
optical_models = {"coupler": DirectionalCoupler, "bias": PhaseBias, "R": Resistor}
simple_net = optical_netlist(("ARM_A,p1", "ARM_A,p2"), ("ARM_B,p1", "ARM_B,p2"))
simple_net["instances"].update(ARM_A={"component": "bias"}, ARM_B={"component": "bias"})
simple_mzm = compile_circuit(simple_net, optical_models, is_complex=True)
static_net = {
    "instances": {
        "GND": {"component": "ground"},
        "MZM": {"component": "mzm"},
        "LASER": {"component": "laser", "settings": {"power": laser_power}},
        "LOAD": {"component": "R", "settings": {"R": 1.0}},
    },
    "connections": {"LASER,p1": "MZM,optical_in", "LASER,p2": "GND,p1", "MZM,optical_out": "LOAD,p1", "LOAD,p2": "GND,p1"},
    "ports": {"output": "LOAD,p1"},
}
static = compile_circuit(static_net, {"mzm": simple_mzm, "laser": OpticalSource, "R": Resistor}, is_complex=True)
phases = jnp.linspace(-np.pi, np.pi, 41)
static_solution = static.dc(params={"MZM~BIAS.phase": phases})
transmission = np.abs(np.asarray(static.port(static_solution, "output"))) ** 2 / laser_power
np.testing.assert_allclose(transmission, (1 - np.cos(phases)) / 2, atol=1e-7)
plt.plot(phases / np.pi, transmission)
plt.plot(-0.5, 0.5, "o", label="Quadrature")
plt.xlabel("Bias phase / pi")
plt.ylabel("Optical transmission")
plt.legend()
plt.show()

## 2. Create the modal transmission lines

For a uniform symmetric pair, use differential and common modes:

$$V_d=V_a-V_b,\quad V_c=(V_a+V_b)/2,\qquad I_d=(I_a-I_b)/2,\quad I_c=I_a+I_b.$$

Two 30 Ω conductor terminations become **60 Ω differential and 15 Ω common mode**. For per-conductor odd/even impedances, use $Z_d=2Z_\mathrm{odd}$ and $Z_c=Z_\mathrm{even}/2$.

`mode_transmission_line` accepts functions, sampled arrays or scalars:

| Input | Unit |
| --- | --- |
| `attenuation_db_per_m` | dB/m |
| `n_eff` — RF phase index | dimensionless |
| `impedance` — possibly complex | Ω |

The factory converts these to a passive RLC line. The example uses **bare-electrode** data; we add PN loading next. If your mode solve already includes the junction R/C, omit that additional electrical loading. Keep the optical phase response.

These illustrative modes are deliberately distinct: the common mode has higher loss, a larger phase index and a different impedance. Only the **loaded differential mode** is chosen to match the optical velocity. Actual modal differences depend on the electrode and return-path geometry; use extracted data for a device prediction.

In [ ]:
frequencies = np.r_[0.0, np.geomspace(1e7, 120e9, 200)]


def attenuation_d(f_hz):
    """Illustrative differential-mode attenuation, in dB/m."""
    x = f_hz / 40e9
    return 250.0 + 150.0 * x**2 / (1 + x**2)


def neff_d(f_hz):
    return 2.2 + 0.0206 / (1 + (f_hz / 40e9) ** 2)


def attenuation_c(f_hz):
    x = f_hz / 25e9
    return 600.0 + 600.0 * x**2 / (1 + x**2)


def neff_c(f_hz):
    return 2.7 + 0.1318 / (1 + (f_hz / 25e9) ** 2)


# An extracted Z(f) array or callable can replace either constant impedance.
line_d = mode_transmission_line(frequencies, attenuation_db_per_m=attenuation_d, n_eff=neff_d, impedance=110.0)
line_c = mode_transmission_line(frequencies, attenuation_db_per_m=attenuation_c, n_eff=neff_c, impedance=45.0)
half_d, half_c = line_d(length=dz / 2), line_c(length=dz / 2)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for label, attenuation, index, line in (
    ("Differential", attenuation_d, neff_d, line_d),
    ("Common", attenuation_c, neff_c, line_c),
):
    axes[0].plot(frequencies / 1e9, attenuation(frequencies), label=label)
    axes[1].plot(frequencies / 1e9, index(frequencies), label=label)
    print(f"{label} fit NRMSE: series {line.series.normalized_rmse:.3%}, shunt {line.shunt.normalized_rmse:.3%}")
axes[0].set(xlabel="Frequency (GHz)", ylabel="Bare-electrode attenuation (dB/m)")
axes[1].set(xlabel="Frequency (GHz)", ylabel="Bare-electrode RF phase index")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Add the junction phase shifters

At every section midpoint, a power-preserving `DifferentialAdapter` converts modal signals to conductor voltages. Physical junction currents load both modal lines through the same adapter.

```text
Differential ─ half line ─┬─ half line ─
                      adapter ─ Rj ─ PN arm A
Common ─────── half line ─┴─ half line ─
                          └──── Rj ─ PN arm B
```

Each PN component combines junction charge $Q_j=C_j(V_\mathrm{anode}-V_\mathrm{cathode})$ with

$$\Delta\phi=\Delta z_\mathrm{mm}[a_1(V_j-V_\mathrm{bias})+a_2(V_j^2-V_\mathrm{bias}^2)].$$

Native half-flight history reads place the interaction at the optical section midpoint. The `bias` parameter sets the phase reference; the electrical DC solve determines the actual junction voltage.

In [ ]:
@component(ports=("anode", "cathode", "o1", "o2"), states=("a_forward", "a_reverse", "m_forward", "m_reverse"))
def PNPhaseShifter(  # noqa: N802
    signals,
    length_um=200.0,
    capacitance=64e-15,
    a1=1.2,
    a2=0.15,
    bias=-1.0,
    loss_db_cm=1.0,
    n_group=4.05,
    neff=2.4,
    wavelength_um=1.55,
):
    """Junction charge, quadratic phase and bidirectional native optical delay."""
    past = signals.at_delay(length_um * n_group / (2 * 2.99792458e14))
    half_t = 10 ** (-loss_db_cm * length_um / 10_000 / 40) * jnp.exp(-1j * np.pi * neff * length_um / wavelength_um)
    voltage = (past.anode - past.cathode).real
    phase = length_um * 1e-3 * (a1 * (voltage - bias) + a2 * (voltage**2 - bias**2))
    b1 = half_t * past.m_reverse * jnp.exp(1j * phase)
    b2 = half_t * past.m_forward * jnp.exp(1j * phase)
    charge = capacitance * (signals.anode - signals.cathode)
    return {
        "o1": signals.a_forward - b1,
        "o2": signals.a_reverse - b2,
        "a_forward": signals.o1 - signals.a_forward - b1,
        "a_reverse": signals.o2 - signals.a_reverse - b2,
        "m_forward": signals.m_forward - half_t * past.a_forward,
        "m_reverse": signals.m_reverse - half_t * past.a_reverse,
    }, {"anode": charge, "cathode": -charge}

In [ ]:
phase_settings = {
    "length_um": dz * 1e6,
    "capacitance": cj_per_m * dz,
    "a1": phase_a1,
    "a2": phase_a2,
    "bias": v_bias,
    "loss_db_cm": optical_loss_db_cm,
    "n_group": n_opt,
    "wavelength_um": wavelength_um,
}
v_plot = np.linspace(-1.8, -0.2, 101)
plt.plot(v_plot, length_m * 1e3 * (phase_a1 * (v_plot - v_bias) + phase_a2 * (v_plot**2 - v_bias**2)))
plt.xlabel("Junction voltage (V)")
plt.ylabel("Full-arm phase shift (rad)")
plt.show()

In [ ]:
def electrode_netlist(
    *, n_sections: int, dz: float, rj_product: float, cj_per_m: float, phase_settings: dict | None = None
) -> dict:
    """Repeat modal half-lines, an adapter and two Rj/junction branches.

    Supplying phase_settings replaces each capacitor with a PN phase shifter
    and connects the optical sections along each arm.
    """
    optical = phase_settings is not None
    net = {"instances": {"GND": {"component": "ground"}}, "connections": {}, "ports": {}}
    ins, con, ports = net["instances"], net["connections"], net["ports"]
    for k in range(n_sections):
        ins[f"X{k}"] = {"component": "adapter"}
        for mode, model in (("d", "d_line"), ("c", "c_line")):
            left, right = f"{mode}L{k}", f"{mode}R{k}"
            ins[left], ins[right] = {"component": model}, {"component": model}
            con[f"{left},p2"] = f"X{k},{mode}"
            con[f"{right},p1"] = f"X{k},{mode}"
            if k:
                con[f"{left},p1"] = f"{mode}R{k - 1},p2"
            else:
                ports[f"{mode}_in"] = f"{left},p1"
            ports[f"{mode}_out"] = f"{right},p2"
        for arm in ("a", "b"):
            resistor, junction = f"R{arm}{k}", f"PS{arm}{k}"
            ins[resistor] = {"component": "R", "settings": {"R": rj_product / dz}}
            ins[junction] = (
                {"component": "pn", "settings": phase_settings} if optical else {"component": "C", "settings": {"C": cj_per_m * dz}}
            )
            positive, negative = ("anode", "cathode") if optical else ("p1", "p2")
            con[f"{resistor},p1"] = f"X{k},{arm}"
            con[f"{resistor},p2"] = f"{junction},{positive}"
            con[f"{junction},{negative}"] = "GND,p1"
            ports[f"junction_{arm}_{k}"] = f"{junction},{positive}"
    if optical:
        for arm in ("a", "b"):
            for k in range(n_sections - 1):
                con[f"PS{arm}{k},o2"] = f"PS{arm}{k + 1},o1"
    return net

In [ ]:
electrode_settings = {"n_sections": n_sections, "dz": dz, "rj_product": rj_product, "cj_per_m": cj_per_m}
models = {
    **optical_models,
    "adapter": DifferentialAdapter,
    "pn": PNPhaseShifter,
    "C": Capacitor,
    "d_line": half_d,
    "c_line": half_c,
}
electrical_net = electrode_netlist(**electrode_settings)
electrical = compile_circuit(electrical_net, models, is_complex=False)

## 4. Check bias and electrical bandwidth

Keep the driver and terminations outside the MZM. `VD` drives the differential mode; `VC` supplies the common bias. Electrode resistance and leakage shift the junction bias from the ideal source/load divider.

Use an electrical view for AC: Circulax's driven AC interface excludes nonlinear mixed optical circuits. This view has the same junction capacitances as the PN components, counted once.

In [ ]:
def electrical_bench():
    return {
        "instances": {
            "GND": {"component": "ground"},
            "MZM": {"component": "mzm"},
            "VD": {"component": "drive", "settings": {"dc": 0.0}},
            "VC": {"component": "drive", "settings": {"dc": 2 * v_bias}},
            "RD": {"component": "R", "settings": {"R": 2 * source_r}},
            "RC": {"component": "R", "settings": {"R": source_r / 2}},
            "LD": {"component": "R", "settings": {"R": 2 * load_r}},
            "LC": {"component": "R", "settings": {"R": load_r / 2}},
        },
        "connections": {
            "VD,p1": "RD,p1",
            "RD,p2": "MZM,d_in",
            "MZM,d_out": "LD,p1",
            "VC,p1": "RC,p1",
            "RC,p2": "MZM,c_in",
            "MZM,c_out": "LC,p1",
            "VD,p2": "GND,p1",
            "VC,p2": "GND,p1",
            "LD,p2": "GND,p1",
            "LC,p2": "GND,p1",
        },
        "ports": {f"junction_{arm}_{k}": f"MZM,junction_{arm}_{k}" for arm in ("a", "b") for k in range(n_sections)},
    }

In [ ]:
bench_models = {"R": Resistor, "drive": WaveformVoltageSource}
rf = compile_circuit(electrical_bench(), {**bench_models, "mzm": electrical}, is_complex=False)
rf_dc = rf.dc()
bias_a = np.array([float(rf.port(rf_dc, f"junction_a_{k}")) for k in range(n_sections)])
bias_b = np.array([float(rf.port(rf_dc, f"junction_b_{k}")) for k in range(n_sections)])
np.testing.assert_allclose(bias_a, bias_b, atol=1e-8)
print(f"Junction bias along either arm: {bias_a.round(3)} V")
ac_frequencies = jnp.linspace(1e9, 40e9, 61)
fig, ax = plt.subplots()
for source, label in (("VD", "Differential"), ("VC", "Common")):
    ac = rf.ac_response(freqs=ac_frequencies, sources={source: 1.0}, y_dc=rf_dc)
    va = np.asarray(rf.port(ac["solution"], f"junction_a_{n_sections - 1}"))
    vb = np.asarray(rf.port(ac["solution"], f"junction_b_{n_sections - 1}"))
    modal_voltage = va - vb if source == "VD" else (va + vb) / 2
    rejected_voltage = (va + vb) / 2 if source == "VD" else va - vb
    assert np.max(abs(rejected_voltage)) < 1e-7
    ax.plot(np.asarray(ac_frequencies) / 1e9, 20 * np.log10(abs(2 * modal_voltage)), label=label)
ax.set(xlabel="Frequency (GHz)", ylabel="Last-junction response, relative to ideal divider (dB)")
ax.legend()
plt.show()

### Check velocity matching

For push-pull operation, match the **loaded differential RF group index** to the optical group index. The common mode need not be velocity matched; its different delay affects the response to common-mode ripple. For dispersive data, $n_{g,\mathrm{RF}}=n_\mathrm{eff}+f\,dn_\mathrm{eff}/df$.

The helper calculates the loaded index from the repeated RLC section, including junction R/C and section discretization. Bare-electrode phase index is lower because junction capacitance slows the RF wave. If your mode data already includes junction loading, compare its group index directly.

In [ ]:
def loaded_group_index(line, frequencies, *, scale: float, dz: float, rj_product: float, cj_per_m: float) -> np.ndarray:
    """Differentiate the Bloch phase of a symmetric, junction-loaded RLC cell.

    scale converts the physical junction to differential (2) or common (0.5)
    mode: R_modal = scale * Rj and C_modal = Cj / scale.
    """
    f = np.asarray(frequencies)
    s_ac = 2j * np.pi * f
    d = dz / 2
    series = np.tile(np.eye(2, dtype=complex), (len(f), 1, 1))
    shunt = series.copy()
    series[:, 0, 1] = line.series.evaluate(f) * d / 2
    shunt[:, 1, 0] = line.shunt.evaluate(f) * d
    half = series @ shunt @ series
    junction = np.tile(np.eye(2, dtype=complex), (len(f), 1, 1))
    junction[:, 1, 0] = s_ac * (cj_per_m * dz / scale) / (1 + s_ac * rj_product * cj_per_m)
    cell = half @ junction @ half
    gamma_loaded = np.arccosh((cell[:, 0, 0] + cell[:, 1, 1]) / 2) / dz
    return 299_792_458.0 * np.gradient(gamma_loaded.imag, 2 * np.pi * f)

In [ ]:
fig, ax = plt.subplots()
for label, line, scale in (("Differential", line_d, 2.0), ("Common", line_c, 0.5)):
    group_index = loaded_group_index(line, ac_frequencies, scale=scale, dz=dz, rj_product=rj_product, cj_per_m=cj_per_m)
    ax.plot(np.asarray(ac_frequencies) / 1e9, group_index, label=f"Loaded RF: {label}")
    design = np.argmin(abs(np.asarray(ac_frequencies) - 20e9))
    walkoff_ps = length_m * (group_index[design] - n_opt) / c0 * 1e12
    print(f"{label} near 20 GHz: loaded group index {group_index[design]:.3f}, optical {n_opt:.3f}; walk-off {walkoff_ps:.2f} ps")
ax.axhline(n_opt, color="k", linestyle="--", label="Optical group index")
ax.set(xlabel="Frequency (GHz)", ylabel="Group index")
ax.legend()
plt.show()

## 5. Assemble the traveling-wave MZM

Replace the simple arms with repeated PN sections. Reuse the splitter, combiner and bias shifter. The resulting subcircuit exposes optical and modal RF input/output ports, plus junction-voltage monitors.

In [ ]:
mzm_net = electrode_netlist(**electrode_settings, phase_settings=phase_settings)
optical = optical_netlist(("PSa0,o1", f"PSa{n_sections - 1},o2"), ("PSb0,o1", f"PSb{n_sections - 1},o2"))
for key in ("instances", "connections", "ports"):
    mzm_net[key].update(optical[key])
mzm = compile_circuit(mzm_net, models, is_complex=True, rtol=1e-9, atol=1e-10)
print("MZM ports:", [p for p in mzm.ports if not p.startswith("junction")])

## 6. Add grating couplers

```text
CW laser → input grating → MZM → output grating → optical load
                            ↑
                differential drive + common bias
```

Register the compiled MZM in the parent's model dictionary. Each grating adds 3 dB loss at 1550 nm. Observe power before and after the output grating so insertion loss stays visible. These grating models add no envelope dispersion.

In [ ]:
link_net = electrical_bench()
link_net["instances"].update(
    {
        "LASER": {"component": "laser", "settings": {"power": laser_power}},
        "GRATING_IN": {
            "component": "grating",
            "settings": {"peak_loss_dB": grating_loss_db, "center_wavelength_nm": 1550.0, "wavelength_nm": 1550.0},
        },
        "GRATING_OUT": {
            "component": "grating",
            "settings": {"peak_loss_dB": grating_loss_db, "center_wavelength_nm": 1550.0, "wavelength_nm": 1550.0},
        },
        "OPTICAL_LOAD": {"component": "R", "settings": {"R": 1.0}},
    }
)
link_net["connections"].update(
    {
        "LASER,p1": "GRATING_IN,grating",
        "LASER,p2": "GND,p1",
        "GRATING_IN,waveguide": "MZM,optical_in",
        "MZM,optical_out": "GRATING_OUT,waveguide",
        "GRATING_OUT,grating": "OPTICAL_LOAD,p1",
        "OPTICAL_LOAD,p2": "GND,p1",
    }
)
link_net["ports"].update(output="OPTICAL_LOAD,p1", mzm_output="MZM,optical_out")

In [ ]:
link_models = {**bench_models, "mzm": mzm, "laser": OpticalSource, "grating": Grating}
link = compile_circuit(link_net, link_models, is_complex=True, rtol=1e-9, atol=1e-10)
op = link.dc()
received_dc = float(abs(link.port(op, "output")) ** 2)
max_received = laser_power * 10 ** (-(2 * grating_loss_db + optical_loss_db_cm * length_m * 100) / 10)
np.testing.assert_allclose(received_dc, max_received / 2, rtol=1e-6)
print(f"Received power at quadrature: {received_dc * 1e3:.4f} mW")

## 7. Set the swing and run the 56 Gbaud transient

Target **4 dB extinction ratio**, defined as $10\log_{10}(P_\mathrm{high}/P_\mathrm{low})$ between the settled outer optical levels. Sweep the differential DC source about quadrature and choose the source swing from this transfer curve. This includes electrode loading and the nonlinear phase law, while keeping the modulation within the nearly linear part of the MZM response.

The sweep uses the common DC bias with ripple off. `differential_vpp` is the full differential **source** swing; each conductor's source swing is half of that. The loaded junction swing is smaller.

In [ ]:
target_er_db = 4.0
swing_vpp = jnp.linspace(0.0, 2.0, 101)
outer_dc = link.dc(params={"VD.dc": jnp.concatenate((-swing_vpp / 2, swing_vpp / 2))})
outer_power = np.abs(np.asarray(link.port(outer_dc, "output"))) ** 2
low, high = np.sort(outer_power.reshape(2, -1), axis=0)
er_sweep_db = 10 * np.log10(high / low)
assert np.all(np.diff(er_sweep_db) > 0)
assert er_sweep_db[-1] > target_er_db

differential_vpp = float(np.interp(target_er_db, er_sweep_db, np.asarray(swing_vpp)))
outer_dc = link.dc(params={"VD.dc": jnp.asarray([-differential_vpp / 2, differential_vpp / 2])})
outer_power = np.sort(np.abs(np.asarray(link.port(outer_dc, "output"))) ** 2)
calibrated_er_db = 10 * np.log10(outer_power[1] / outer_power[0])
np.testing.assert_allclose(calibrated_er_db, target_er_db, atol=0.01)
print(f"Differential source: {differential_vpp:.3f} Vpp; per conductor: {differential_vpp / 2:.3f} Vpp")
print(f"Settled outer powers: {outer_power * 1e3} mW; ER: {calibrated_er_db:.3f} dB")

Use PRBS7 after a 16-UI constant preamble, with 0.2-UI edges. Set `common_ripple_vpp=0` to remove the common-source ripple.

The transient initializes from the time-zero source levels. Native history reads then propagate the optical fields and apply the local voltage-dependent phase. Output power is the squared field magnitude. Bandwidth and ripple can change the measured dynamic extinction ratio from its DC target.

In [ ]:
def prbs_drive(baud, *, preamble_bits=16, vpp=4.0):
    """Return a PRBS7 pattern and 0.2-UI-edge differential PWL drive."""
    ui = 1 / baud
    state, pattern = 0x7F, []
    for _ in range(127):
        pattern.append(state & 1)
        state = (state >> 1) | (((state ^ (state >> 1)) & 1) << 6)
    if state != 0x7F or sum(pattern) != 64:
        msg = "Invalid PRBS7 period or balance"
        raise ValueError(msg)
    bits = np.r_[np.full(preamble_bits, pattern[0]), pattern]
    levels = vpp * (bits - 0.5)
    times, values = [0.0], [float(levels[0])]
    for k in range(1, len(bits)):
        if levels[k] != levels[k - 1]:
            times.extend([k * ui, (k + 0.2) * ui])
            values.extend([float(levels[k - 1]), float(levels[k])])
    times.append(len(bits) * ui)
    values.append(float(levels[-1]))
    return bits, np.asarray(times), np.asarray(values)

In [ ]:
baud, samples_per_ui = 56e9, 32
ui = 1 / baud
sample_dt, integration_dt = ui / samples_per_ui, ui / (2 * samples_per_ui)
preamble_bits, common_ripple_vpp = 16, 0.2
bits, pwl_t, pwl_v = prbs_drive(baud, preamble_bits=preamble_bits, vpp=differential_vpp)
t_stop = len(bits) * ui
transient_net = copy.deepcopy(link_net)
transient_net["instances"]["VD"]["settings"] = {"kind": WAVE_PWL, "pwl_t": jnp.asarray(pwl_t), "pwl_v": jnp.asarray(pwl_v)}
transient_net["instances"]["VC"]["settings"] = {
    "kind": WAVE_SIN if common_ripple_vpp else WAVE_DC,
    "dc": 2 * v_bias,
    "offset": 2 * v_bias,
    "amplitude": common_ripple_vpp / 2,
    "freq": 10e9,
    "delay": preamble_bits * ui,
}
transient_link = compile_circuit(transient_net, link_models, is_complex=True, rtol=1e-9, atol=1e-10)

In [ ]:
sample_times = jnp.arange(len(bits) * samples_per_ui + 1) * sample_dt
started = time.perf_counter()
solution = transient_link.transient(
    t0=0.0,
    t1=t_stop,
    dt0=integration_dt,
    saveat=sample_times,
    max_steps=int(np.ceil(t_stop / integration_dt)) + 20,
    throw=True,
)
solution.ys.block_until_ready()
power = np.abs(np.asarray(transient_link.port(solution.ys, "output"))) ** 2
mzm_power = np.abs(np.asarray(transient_link.port(solution.ys, "mzm_output"))) ** 2
np.testing.assert_allclose(power, mzm_power * 10 ** (-grating_loss_db / 10), rtol=1e-6, atol=1e-12)
assert np.all(np.isfinite(power))
assert power.max() <= max_received * 1.001
print(f"Transient: {time.perf_counter() - started:.1f} s; {solution.stats['num_steps']} steps")

## 8. Inspect the eye

Check both junction voltages for push-pull operation, then fold two-UI windows after the preamble and first eight data bits. Start each window half a UI into a symbol: the 0–2 UI view then shows one full eye in the middle and half an eye at each side. Remove only the known optical flight time; RF walk-off and distortion remain.

The eye shows normalized power and received mW, including grating and waveguide loss. Detection is ideal, with no receiver noise or equalization.

In [ ]:
def fold_eye(times, power, *, baud, samples_per_ui, preamble_bits, flight_time):
    """Fold two UI with one central eye and half an eye at each edge."""
    ui = 1 / baud
    n_bits = (len(times) - 1) // samples_per_ui
    eye_ui = np.arange(2 * samples_per_ui + 1) / samples_per_ui
    fold_bits = np.arange(preamble_bits + 8, n_bits - 5)
    # Start at mid-symbol: center one full eye in the two-UI display.
    query = flight_time + (fold_bits[:, None] + 0.5 + eye_ui[None, :]) * ui
    if query.max() >= float(times[-1]):
        msg = "Eye windows extend beyond the simulated record"
        raise ValueError(msg)
    traces = np.interp(query.ravel(), times, power).reshape(query.shape)
    return eye_ui, traces

In [ ]:
optical_flight = length_m * n_opt / c0
clock = (np.asarray(sample_times) - optical_flight) / ui
window = (clock >= preamble_bits + 16) & (clock <= preamble_bits + 26)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for arm in ("a", "b"):
    voltage = np.asarray(transient_link.port(solution.ys, f"junction_{arm}_{n_sections - 1}")).real
    axes[0].plot(clock[window], voltage[window], label=f"Arm {arm.upper()}")
axes[0].set(xlabel="Source-referred time (UI)", ylabel="Last junction voltage (V)")
axes[0].legend()
axes[1].plot(clock[window], power[window] * 1e3, label="Received")
axes[1].plot(clock[window], mzm_power[window] * 1e3, "--", label="Before output grating")
axes[1].set(xlabel="Source-referred time (UI)", ylabel="Optical power (mW)")
axes[1].legend()
eye_ui, eye = fold_eye(
    np.asarray(sample_times),
    power,
    baud=baud,
    samples_per_ui=samples_per_ui,
    preamble_bits=preamble_bits,
    flight_time=optical_flight,
)
axes[2].plot(eye_ui, (eye / max_received).T, color="C0", alpha=0.15, lw=0.8)
axes[2].set(xlabel="Time (UI)", ylabel="Normalized received power", xlim=(0, 2), ylim=(-0.02, 1.02))
axes[2].secondary_yaxis("right", functions=(lambda x: x * max_received * 1e3, lambda x: x / (max_received * 1e3))).set_ylabel(
    "Received power (mW)"
)
fig.suptitle("56 Gbaud push-pull MZM with common-mode ripple")
plt.tight_layout()
plt.show()

## 9. Drive PAM4 with timing jitter

Keep the same **56 Gbaud** circuit and calibrated **4 dB outer-level ER**. PAM4 uses four equally spaced differential source voltages, `differential_vpp * [-1/2, -1/6, 1/6, 1/2]`, carrying two bits per symbol (**112 Gb/s**, before coding). The smaller swing reduces transfer-curve compression; the nonlinear phase law remains active and we apply no predistortion.

Compare the same seeded 128-symbol sequence with and without **transmitter timing jitter**:

- `random_jitter_ui`: Gaussian RMS displacement, clipped at ±3 RMS.
- `periodic_jitter_ui`: peak sinusoidal displacement at `periodic_jitter_hz`.

One UI is 17.86 ps. The defaults give 0.45 ps RMS random jitter before clipping and 1.07 ps peak periodic jitter. The helper shifts each transition's complete 0.2-UI ramp. Jitter is applied to the electrical source **before** circuit simulation, so it propagates through the electrodes and nonlinear phase shifters. The constant preamble and common-mode ripple are retained.

Increase `n_symbols` for denser eyes. This short record illustrates distortion; it does not estimate BER or rare jitter tails.

In [ ]:
def pam4_drive(
    baud,
    *,
    n_symbols=128,
    preamble_symbols=16,
    vpp=4.0,
    random_jitter_ui=0.025,
    periodic_jitter_ui=0.06,
    periodic_jitter_hz=2e9,
    seed=7,
):
    """Return seeded PAM4 symbols and a PWL drive with jittered transition times.

    Random jitter is Gaussian with the specified RMS in UI, clipped at ±3 RMS.
    Periodic jitter is a sinusoidal time displacement with peak amplitude in UI.
    Both shift complete 0.2-UI ramps, preserving their duration and voltage levels.
    """
    rng = np.random.default_rng(seed)
    data = rng.integers(0, 4, n_symbols)
    symbols = np.r_[np.full(preamble_symbols, data[0]), data]
    levels = vpp * (symbols / 3 - 0.5)
    ui = 1 / baud
    edges = np.arange(1, len(symbols)) * ui
    random = np.clip(rng.normal(size=len(edges)), -3, 3) * random_jitter_ui
    periodic = periodic_jitter_ui * np.sin(2 * np.pi * periodic_jitter_hz * edges)
    shifts = ui * (random + periodic)
    shifts[edges < preamble_symbols * ui] = 0.0
    times, values = [0.0], [float(levels[0])]
    for k, start in enumerate(edges + shifts, start=1):
        if levels[k] != levels[k - 1]:
            times.extend([start, start + 0.2 * ui])
            values.extend([float(levels[k - 1]), float(levels[k])])
    times.append(len(symbols) * ui)
    values.append(float(levels[-1]))
    if np.any(np.diff(times) <= 0):
        msg = "Jitter is too large: PWL edges overlap or leave the simulation interval"
        raise ValueError(msg)
    return symbols, np.asarray(times), np.asarray(values)

In [ ]:
n_symbols = 128
random_jitter_ui, periodic_jitter_ui, periodic_jitter_hz = 0.025, 0.06, 2e9
pam4_results = {}
for label, jitter_scale in (("No timing jitter", 0.0), ("With timing jitter", 1.0)):
    symbols, drive_t, drive_v = pam4_drive(
        baud,
        n_symbols=n_symbols,
        preamble_symbols=preamble_bits,
        vpp=differential_vpp,
        random_jitter_ui=jitter_scale * random_jitter_ui,
        periodic_jitter_ui=jitter_scale * periodic_jitter_ui,
        periodic_jitter_hz=periodic_jitter_hz,
        seed=7,
    )
    pam4_net = copy.deepcopy(transient_net)
    pam4_net["instances"]["VD"]["settings"].update(pwl_t=jnp.asarray(drive_t), pwl_v=jnp.asarray(drive_v))
    pam4_link = compile_circuit(pam4_net, link_models, is_complex=True, rtol=1e-9, atol=1e-10)
    pam4_times = jnp.arange(len(symbols) * samples_per_ui + 1) * sample_dt
    started = time.perf_counter()
    pam4_solution = pam4_link.transient(
        t0=0.0,
        t1=len(symbols) * ui,
        dt0=integration_dt,
        saveat=pam4_times,
        max_steps=2 * len(symbols) * samples_per_ui + 20,
        throw=True,
    )
    pam4_solution.ys.block_until_ready()
    received = np.abs(np.asarray(pam4_link.port(pam4_solution.ys, "output"))) ** 2
    assert np.all(np.isfinite(received))
    assert received.max() <= max_received * 1.001
    source = np.interp(np.asarray(pam4_times), drive_t, drive_v)
    pam4_results[label] = source, received
    settled_symbols = np.arange(preamble_bits + 8, len(symbols) - 5)
    sample_t = optical_flight + (settled_symbols + 0.75) * ui
    sampled_power = np.interp(sample_t, np.asarray(pam4_times), received)
    outer_levels = [np.median(sampled_power[symbols[settled_symbols] == level]) for level in (0, 3)]
    dynamic_er_db = abs(10 * np.log10(outer_levels[1] / outer_levels[0]))
    print(f"{label}: sampled outer-level ER {dynamic_er_db:.2f} dB")
    print(f"{label}: {time.perf_counter() - started:.1f} s; {pam4_solution.stats['num_steps']} steps")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for row, (label, (source, received)) in enumerate(pam4_results.items()):
    for col, (signal, delay) in enumerate(((source, 0.0), (received * 1e3, optical_flight))):
        eye_ui, traces = fold_eye(
            np.asarray(pam4_times),
            signal,
            baud=baud,
            samples_per_ui=samples_per_ui,
            preamble_bits=preamble_bits,
            flight_time=delay,
        )
        axes[row, col].plot(eye_ui, traces.T, color=f"C{row}", alpha=0.15, lw=0.8)
        axes[row, col].set(title=label, xlim=(0, 2))
    axes[row, 0].set_ylabel("Differential source (V)")
    axes[row, 1].set(ylabel="Received power (mW)", ylim=(-0.02 * max_received * 1e3, 1.02 * max_received * 1e3))
for ax in axes[-1]:
    ax.set_xlabel("Time (UI)")
fig.suptitle("56 Gbaud PAM4: electrical source and received optical eyes")
plt.tight_layout()
plt.show()

Fold both cases against the same ideal symbol clock, with the same half-UI window offset used above. Each 2-UI plot shows a full eye in the center and half an eye at each side. Remove only the optical flight time from the received signal, so timing jitter and RF walk-off remain visible. The three openings are the PAM4 eyes. At this reduced swing, their heights are similar; residual differences include the nonlinear transfer and RF distortion. Compare the rows to see timing jitter broaden the transitions. The printed dynamic ER uses median outer-level powers sampled 0.75 UI into each settled symbol. Receiver noise and clock recovery are absent.

Try changing attenuation, RF index, junction R/C, common-mode ripple or load resistance. Increase `n_sections` and reduce `integration_dt` to check convergence.

For a line without internal junction taps, see [the reduced RLGC line example](../electrical/rlgc_transmission_line.ipynb).